# Hotel Recommendation System

## Project Objective

The objective of this project is to build a hotel recommendation system using collaborative filtering techniques. The system recommends hotels to users based on booking patterns of similar users.

In [1]:
import pandas as pd
import numpy as np

## Dataset Loading

The cleaned hotel, user, and flight datasets are loaded into the notebook for analysis and recommendation generation.

In [2]:
hotels = pd.read_csv("../dataset/cleaned_hotels.csv")
users = pd.read_csv("../dataset/cleaned_users.csv")
flights = pd.read_csv("../dataset/cleaned_flights.csv")

print("Hotels Shape:", hotels.shape)
print("Users Shape:", users.shape)
print("Flights Shape:", flights.shape)

Hotels Shape: (40552, 8)
Users Shape: (1340, 5)
Flights Shape: (60354, 10)


## Dataset Preview

The first few records of the datasets are displayed to understand the available information and structure.

In [3]:
hotels.head()

,travelCode,userCode,name,place,days,price,total,date
0,0,0,Hotel A,Florianopolis (SC),4,313.02,1252.08,09/26/2019
1,2,0,Hotel K,Salvador (BH),2,263.41,526.82,10/10/2019
2,7,0,Hotel K,Salvador (BH),3,263.41,790.23,11/14/2019
3,11,0,Hotel K,Salvador (BH),4,263.41,1053.64,12/12/2019
4,13,0,Hotel A,Florianopolis (SC),1,313.02,313.02,12/26/2019


In [4]:
users.head()

,code,company,name,gender,age
0,0,4You,Roy Braun,male,21
1,1,4You,Joseph Holsten,male,37
2,2,4You,Wilma Mcinnis,female,48
3,3,4You,Paula Daniel,female,23
4,4,4You,Patricia Carson,female,44


In [5]:
flights.head()

,travelCode,userCode,from,to,flightType,price,time,distance,agency,date
0,0,0,Recife (PE),Florianopolis (SC),firstClass,1434.38,1.76,676.53,FlyingDrops,09/26/2019
1,0,0,Florianopolis (SC),Recife (PE),firstClass,1292.29,1.76,676.53,FlyingDrops,09/30/2019
2,1,0,Brasilia (DF),Florianopolis (SC),firstClass,1487.52,1.66,637.56,CloudFy,10/03/2019
3,1,0,Florianopolis (SC),Brasilia (DF),firstClass,1127.36,1.66,637.56,CloudFy,10/04/2019
4,2,0,Aracaju (SE),Salvador (BH),firstClass,1684.05,2.16,830.86,CloudFy,10/10/2019


## User-Hotel Interaction Matrix

A user-hotel matrix is created using cross-tabulation. This matrix represents the interaction frequency between users and hotels.

In [6]:
user_hotel = pd.crosstab(
    hotels["userCode"],
    hotels["name"]
)

print(user_hotel.shape)

user_hotel.head()

(1310, 9)


name,Hotel A,Hotel AF,Hotel AU,Hotel BD,Hotel BP,Hotel BW,Hotel CB,Hotel K,Hotel Z
userCode,,,,,,,,,
0,3,4,2,4,1,2,1,7,3
1,0,1,0,0,1,0,0,0,0
2,6,2,3,2,2,7,3,5,6
3,10,6,7,11,2,9,7,6,2
4,7,6,7,5,3,6,11,7,4


## Similarity Calculation

Cosine Similarity is used to calculate similarity between users based on their hotel booking history.

In [7]:
from sklearn.metrics.pairwise import cosine_similarity

similarity = cosine_similarity(user_hotel)

print(similarity.shape)

(1310, 1310)


## Recommendation Function

The recommendation engine identifies users with similar booking behavior and suggests hotels they have visited but the target user has not.

In [8]:
def recommend_hotels(user_id, top_n=5):

    if user_id not in user_hotel.index:
        return []

    user_idx = list(user_hotel.index).index(user_id)

    scores = similarity[user_idx]

    similar_users = np.argsort(scores)[::-1][1:6]

    hotels_seen = set(
        user_hotel.loc[user_id][
            user_hotel.loc[user_id] > 0
        ].index
    )

    recommendations = []

    for idx in similar_users:

        sim_user = user_hotel.index[idx]

        sim_hotels = set(
            user_hotel.loc[sim_user][
                user_hotel.loc[sim_user] > 0
            ].index
        )

        recommendations.extend(
            list(sim_hotels - hotels_seen)
        )

    return list(set(recommendations))[:top_n]

## Recommendation Testing

The recommendation function is tested using a sample user ID.

In [9]:
recommend_hotels(0)

[]

## Model Serialization

The recommendation matrix is saved using Joblib so it can be reused during deployment.

In [9]:
import joblib

joblib.dump(
    user_hotel,
    "../models/recommendation_model.pkl"
)

print("Recommendation Model Saved Successfully")

Recommendation Model Saved Successfully


## MLflow Experiment Tracking

MLflow is integrated to track experiment parameters and metrics related to the recommendation model.

In [10]:
import joblib

joblib.dump(
    user_hotel,
    "../models/recommendation_model.pkl"
)

print("Recommendation Model Saved Successfully")

Recommendation Model Saved Successfully


## MLflow Experiment Tracking

MLflow is integrated to track experiment parameters and metrics related to the recommendation model.

In [11]:
import mlflow

mlflow.set_experiment(
    "Hotel_Recommendation_System"
)

with mlflow.start_run():

    mlflow.log_param(
        "algorithm",
        "Cosine Similarity"
    )

    mlflow.log_metric(
        "users",
        user_hotel.shape[0]
    )

    mlflow.log_metric(
        "hotels",
        user_hotel.shape[1]
    )

print("Tracking Done")

Tracking Done


## Project Summary

Successfully developed a Hotel Recommendation System using Collaborative Filtering and Cosine Similarity.

### Features Implemented

- Data Loading and Exploration
- User-Hotel Interaction Matrix
- Cosine Similarity Recommendation Engine
- Recommendation Testing
- Model Serialization using Joblib
- MLflow Experiment Tracking
- FastAPI Deployment
- Streamlit Frontend
- Docker Containerization

### Technologies Used

- Python
- Pandas
- NumPy
- Scikit-Learn
- Joblib
- MLflow
- FastAPI
- Streamlit
- Docker